In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

processed_dir = Path.cwd().parent / "data" / "processed"

long_df = pd.read_csv(processed_dir / "stray_dogs_tidy_long.csv")
wide_df = pd.read_csv(processed_dir / "stray_dogs_tidy_wide.csv")

Summary

The dataset is small and other sources would need to be pulled in to draw more conclusions, however some have been made.

The plot below shows `dogs_seized` declined by ~74% from 2016 to 2021, then remained fairly consistent from 2022-2024. It seems fair to suggest that this decline could be from the introduction of compulsory microchipping in England in April 2016, which likely made reuniting owners with their pets a lot easier.

Separately, the sum of the six recorded outcome categories consistently falls short of `dogs_seized` each year (see `unaccounted (seized - outcomes)`. My hypothesis here is that there is an additional unrecorded category - 'rehomed'. This is consistent with Dogs Trust's national 2023 Stray Dog Survey, which reported roughly a fifth of all dogs handled by local authority wardens were passed on to welfare organisations for rehoming, though this is a national figure rather than Rochdale-specific. The 2022–2023 rebound in the gap could reflect post-COVID pressure on rehoming capacity, though I have no data to confirm this.

In [ ]:
component_metrics = [
    "dogs_seized",
    "dogs_returned_by_warden",
    "dogs_returned_by_kennel_provider",
    "dogs_retained_by_finder",
    "dogs_euthanised",
]

yearly_totals = (
    long_df[long_df["metric"].isin(component_metrics)]
    .groupby(["year", "metric"])["value"]
    .sum()
    .unstack("metric")
)

outcome_only_metrics = [
    "dogs_returned_by_warden",
    "dogs_returned_by_kennel_provider",
    "dogs_retained_by_finder",
    "dogs_euthanised",
]

sum_of_outcomes = yearly_totals[outcome_only_metrics].sum(axis=1)
unaccounted = (yearly_totals["dogs_seized"] - sum_of_outcomes).rename("unaccounted (seized - outcomes)")

fig, ax = plt.subplots(figsize=(9, 5))
yearly_totals.plot(ax=ax, marker="o")
unaccounted.plot(ax=ax, marker="o", linestyle="--", linewidth=2.5, color="black")
ax.set_xlabel("Year")
ax.set_ylabel("Number of dogs")
ax.set_title("Each outcome category by year (raw counts)")
ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(figures_dir / "component_trends_raw.png", dpi=150)
plt.show()

In [ ]:
seized = long_df[long_df["metric"] == "dogs_seized"]

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
pivot = seized.pivot(index="year", columns="month_num", values="value")

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(12))
ax.set_xticklabels(month_names)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("Month")
ax.set_ylabel("Year")
ax.set_title("Dogs seized by month and year")
fig.colorbar(im, ax=ax, label="Dogs seized")
plt.tight_layout()

figures_dir = Path.cwd().parent / "outputs" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(figures_dir / "seasonality_heatmap.png", dpi=150)
plt.show()